In [7]:
# Define la estructura de cada nodo para almacenar un pedido
class NodoPedido: 
    # Constructor que inicializa las propiedades esenciales de un pedido
    def __init__(self, id_pedido: int, tipo: str, descripcion: str): 
        # Guarda el número de identificación único del pedido
        self.id_pedido = id_pedido      
        # Guarda la categoría del pedido (ej. reclamo o estándar)
        self.tipo = tipo                
        # Guarda el texto explicativo sobre el pedido
        self.descripcion = descripcion
        # Puntero para enlazar con el siguiente nodo en la estructura
        self.siguiente = None           

# Define la clase principal para administrar el flujo de pedidos
class GestorPedidos: 
    # Constructor que prepara las estructuras de datos vacías
    def __init__(self): 
        # Referencia al primer elemento de la estructura de cola
        self.cola_frente = None 
        # Referencia al último elemento de la estructura de cola
        self.cola_fin = None 
        # Referencia al elemento superior de la estructura de pila
        self.pila_tope = None

    # Comprueba si la estructura de cola no tiene elementos
    def cola_esta_vacia(self):
        # Evalúa si la referencia del frente apunta a la nada
        return self.cola_frente is None
    
    # Comprueba si la estructura de pila no tiene elementos
    def pila_esta_vacia(self):
        # Evalúa si la referencia del tope apunta a la nada
        return self.pila_tope is None
    
    # Busca si un identificador ya está registrado en alguna estructura
    def existe_id_registro(self, id_a_buscar):
        # Comienza la búsqueda desde el principio de la cola
        actual = self.cola_frente
        # Recorre la cola mientras existan nodos disponibles
        while actual is not None:
            # Compara el identificador del nodo con el buscado
            if actual.id_pedido == id_a_buscar:
                # Retorna verdadero si encuentra la coincidencia
                return True
            # Avanza al siguiente elemento en el enlace de la cola
            actual = actual.siguiente
        
        # Comienza la búsqueda desde el tope de la pila
        actual = self.pila_tope
        # Recorre la pila mientras existan nodos disponibles
        while actual is not None:
            # Compara el identificador del nodo con el buscado
            if actual.id_pedido == id_a_buscar:
                # Retorna verdadero si encuentra la coincidencia
                return True
            # Avanza al siguiente elemento en el enlace de la pila
            actual = actual.siguiente
        
    # Agrega un nuevo pedido al sistema según su clasificación
    def registrar_pedido(self, id_pedido: int, tipo: str, descripcion: str) -> None:
        # Verifica si el identificador ya fue usado antes
        if self.existe_id_registro(id_pedido):
            print(f"Error: El ID de pedido {id_pedido} ya existe.")
            # Cancela el proceso para evitar duplicados
            return
        # Crea una nueva instancia de nodo con los datos provistos
        nuevo_pedido = NodoPedido(id_pedido, tipo, descripcion)

        # Evalúa si el caso corresponde a un reclamo prioritario
        if tipo == "RECLAMO":
            # Conecta el nuevo nodo al nodo que estaba en el tope de la pila
            nuevo_pedido.siguiente = self.pila_tope
            # Actualiza el tope de la pila para que apunte al nuevo nodo
            self.pila_tope = nuevo_pedido
            print(f"RECLAMO con ID '{self.pila_tope.id_pedido}' apilado: '{self.pila_tope.descripcion}'")
            
        # Evalúa si corresponde a una solicitud de tipo ordinaria
        elif tipo in ["ESTÁNDAR", "ESTANDAR"]:
            # Verifica si es el primer elemento en ingresar a la cola
            if self.cola_esta_vacia():
                # Define el nuevo nodo como el inicio de la cola
                self.cola_frente = nuevo_pedido
                # Define el mismo nodo como el final de la cola
                self.cola_fin = nuevo_pedido
            # Si ya hay elementos en la cola realiza el enlace al final
            else:
                # Conecta el último nodo actual con el nuevo nodo creado
                self.cola_fin.siguiente = nuevo_pedido
                # Actualiza el indicador del final de la cola al nuevo nodo
                self.cola_fin = nuevo_pedido
            print(f"PEDIDO con ID '{self.cola_fin.id_pedido}' encolado: '{self.cola_fin.descripcion}'")

    # Procesa y extrae el elemento que corresponde atender según prioridad
    def despachar_siguiente(self) -> NodoPedido:
        # Verifica si no existen pedidos pendientes en ninguna estructura
        if self.cola_esta_vacia() and self.pila_esta_vacia():
            print("No hay nada que despachar")
            # Retorna vacío al no haber tareas pendientes
            return None
        
        # Da paso a la cola si la pila de reclamos prioritarios está vacía
        if self.pila_tope is None and self.cola_frente is not None:
            # Separa el elemento del frente de la cola para despacharlo
            pedido_despachado = self.cola_frente
            # Mueve el inicio de la cola hacia el nodo posterior
            self.cola_frente = self.cola_frente.siguiente
        # Da prioridad de despacho a los elementos de la pila si existen
        else:
            # Separa el elemento del tope de la pila para despacharlo
            pedido_despachado = self.pila_tope
            # Mueve el tope de la pila hacia el nodo posterior
            self.pila_tope = self.pila_tope.siguiente

        # Verifica si la cola quedó vacía tras la extracción anterior
        if self.cola_frente is None: 
            # Asegura que el indicador final de la cola también quede en nada
            self.cola_fin = None
        # Devuelve el nodo completo que fue procesado
        return pedido_despachado 

    # Mueve de golpe todos los elementos de la cola hacia la pila
    def activar_contingencia_masiva(self) -> None:
        # Verifica si no hay elementos en la cola para transferir
        if self.cola_esta_vacia():
            print("No hay pedidos para transferir")
            # Termina la ejecución al no haber elementos
            return
        # Inicializa el contador para registrar los traslados exitosos
        en_contingencia = 0
        # Ejecuta el bucle hasta vaciar por completo los nodos de la cola
        while self.cola_frente is not None:
            # Toma temporalmente el primer nodo disponible en la cola
            nodo_saliente = self.cola_frente
            # Desplaza el frente de la cola al nodo posterior
            self.cola_frente = self.cola_frente.siguiente

            # Conecta el nodo extraído por encima del tope actual de la pila
            nodo_saliente.siguiente = self.pila_tope
            # Define el nodo extraído como el nuevo tope oficial de la pila
            self.pila_tope = nodo_saliente
            # Modifica la etiqueta de origen a una de contingencia
            nodo_saliente.tipo = "CONTINGENCIA"
            # Incrementa en uno el registro de elementos trasladados
            en_contingencia += 1

        # Limpia la referencia de fin de la cola porque quedó vacía
        self.cola_fin = None
        print(f"Contingencia masiva activada. {en_contingencia} pedidos transferidos a la pila de reclamos.")

    # Muestra en pantalla el contenido de la estructura elegida
    def mostrar_pila_cola(self, criterio):
        # Valida si se solicitó ver la estructura de cola
        if criterio == "cola":
            # Detiene el flujo si la estructura no contiene datos
            if self.cola_esta_vacia():
                print("No hay pedidos que mostrar")
                # Sale del método de manera anticipada
                return
            
            # Establece el inicio del recorrido en el frente de la cola
            actual = self.cola_frente
            print("Mostrando pedidos - COLA")
            # Ciclo para iterar por todos los nodos enlazados en la cola
            while actual is not None:
                print(f"ID pedido: {actual.id_pedido}, Tipo: {actual.tipo}, Descripción: {actual.descripcion}")
                # Desplaza el recorrido hacia el nodo enlazado que sigue
                actual = actual.siguiente

        # Valida si se solicitó ver la estructura de pila
        elif criterio == "pila":
            # Detiene el flujo si la estructura no contiene datos
            if self.pila_esta_vacia():
                print("No hay reclamos que mostrar")
                # Sale del método de manera anticipada
                return
            # Establece el inicio del recorrido en el tope de la pila
            actual = self.pila_tope
            print("Mostrando reclamos - PILA")
            # Ciclo para iterar por todos los nodos enlazados en la pila
            while actual is not None:
                print(f"ID pedido: {actual.id_pedido}, Tipo: {actual.tipo}, Descripción: {actual.descripcion}")
                # Desplaza el recorrido hacia el nodo enlazado que sigue
                actual = actual.siguiente

In [8]:
# Instancia el objeto principal para controlar todo el flujo del sistema
pedido = GestorPedidos()
# Define la función principal que maneja el menú interactivo por consola
def interfaz():
    # Inicia un ciclo infinito para mantener el programa activo hasta que se decida salir
    while True:
        # Menú visual de las opciones disponibles
        print("\n MENÚ GESTIÓN DE PEDIDOS DELIVERY ")
        print("1. Registrar nuevo pedido")
        print("2. Despachar siguiente pedido")
        print("3. Activar contingencia masiva")
        print("4. Mostrar pedidos - Cola")
        print("5. Mostrar reclamos o pedidos en contingencia - Pila")
        print("6. Salir del programa")
        # Captura la opción del usuario en formato de texto
        opcion = input("Seleccione una opción: ")

        # Evalúa si el usuario seleccionó la opción de registro
        if opcion == "1":
            # Inicia un ciclo para forzar el ingreso correcto de los datos del pedido
            while True:
                # Bloque de seguridad para capturar errores de escritura en los datos numéricos
                try:
                    # Convierte a entero la entrada de texto del identificador
                    id_pedido = int(input("Ingrese el ID del pedido: "))
                    # Limpia espacios sobrantes y convierte a mayúsculas el tipo ingresado
                    tipo = input("Ingrese el tipo de pedido (ESTÁNDAR/RECLAMO): ").strip().upper()
                    # Captura y limpia los espacios en los extremos de la descripción
                    descripcion = input("Ingrese la descripción del pedido: ").strip()
                    
                    # Valida que cumpla las reglas de negocio antes de registrarlo
                    if id_pedido > 0 and tipo in ["ESTÁNDAR", "RECLAMO", "ESTANDAR"] and descripcion:
                        # Envía la información depurada al método de la instancia global
                        pedido.registrar_pedido(id_pedido, tipo, descripcion)
                        # Rompe el ciclo interno al procesar los datos de forma exitosa
                        break
                    # Ejecuta esta sección si los datos están vacíos o no cumplen con los tipos válidos
                    else:
                        print("Datos inválidos. Intente de nuevo.")
                # Atrapa la excepción si el identificador no se puede transformar a número entero
                except ValueError:
                    print("Asegúrese de ingresar correctamente el tipo de dato.")
                    # Vuelve a iniciar el bucle interno de captura de datos

        # Evalúa si se seleccionó la opción de procesar despacho
        elif opcion == "2":
            # Ejecuta la función de despacho y guarda el nodo resultante
            despachado = pedido.despachar_siguiente()
            # Valida si la función devolvió algún nodo válido para atender
            if despachado is not None:
                print(f"DESPACHANDO: ID {despachado.id_pedido}, Tipo: {despachado.tipo}, Descripción: {despachado.descripcion}")

        # Evalúa si se solicitó la migración por emergencia masiva
        elif opcion == "3":
            # Llama al método encargado de volcar la cola en la pila
            pedido.activar_contingencia_masiva()

        # Evalúa si se solicitó revisar el listado ordinario
        elif opcion == "4":
            # Invoca la impresión filtrando específicamente por la estructura de cola
            pedido.mostrar_pila_cola("cola")

        # Evalúa si se solicitó revisar el listado de incidencias
        elif opcion == "5":
            # Invoca la impresión filtrando específicamente por la estructura de pila
            pedido.mostrar_pila_cola("pila")

        # Evalúa si el usuario decidió dar por terminado el flujo
        elif opcion == "6":
            print("Saliendo del programa de delivery")
            # Rompe el ciclo principal para finalizar la ejecución de la consola
            break
        
        # Captura cualquier entrada de texto que no corresponda al menú
        else:
            print("\nOpción no válida.")
# Llama a la función para poner en marcha la interfaz del sistema
interfaz()


 MENÚ GESTIÓN DE PEDIDOS DELIVERY 
1. Registrar nuevo pedido
2. Despachar siguiente pedido
3. Activar contingencia masiva
4. Mostrar pedidos - Cola
5. Mostrar reclamos o pedidos en contingencia - Pila
6. Salir del programa
No hay nada que despachar

 MENÚ GESTIÓN DE PEDIDOS DELIVERY 
1. Registrar nuevo pedido
2. Despachar siguiente pedido
3. Activar contingencia masiva
4. Mostrar pedidos - Cola
5. Mostrar reclamos o pedidos en contingencia - Pila
6. Salir del programa
No hay pedidos para transferir

 MENÚ GESTIÓN DE PEDIDOS DELIVERY 
1. Registrar nuevo pedido
2. Despachar siguiente pedido
3. Activar contingencia masiva
4. Mostrar pedidos - Cola
5. Mostrar reclamos o pedidos en contingencia - Pila
6. Salir del programa
No hay pedidos que mostrar

 MENÚ GESTIÓN DE PEDIDOS DELIVERY 
1. Registrar nuevo pedido
2. Despachar siguiente pedido
3. Activar contingencia masiva
4. Mostrar pedidos - Cola
5. Mostrar reclamos o pedidos en contingencia - Pila
6. Salir del programa
No hay reclamos que 